# Tuya Device Binding via OpenAPI

Binds a TuyaLink device (registered via Device Management → Register Device) to a
SmartLife home, bypassing the Tuya IoT Platform UI device-pool limit.

## Prerequisites
- Tuya IoT Platform project with **Cloud Development** access
- Project **Access ID** and **Access Secret** (Cloud → Overview → Authorization Key)
- Device's **authorization/bind code** (from Device Management → your device → Bind Code)
- Your SmartLife account's **UID** (see cell below for how to find it)
- The **home_id** of the SmartLife home to bind to


In [ ]:
%pip install tuya-connector-python python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (or any parent)
# Create tuya/.env with the following keys:
#   TUYA_ACCESS_ID=...
#   TUYA_ACCESS_KEY=...
#   TUYA_BASE_URL=https://openapi.tuyaeu.com
#   TUYA_DEVICE_ID=...
#   TUYA_BIND_CODE=         # optional
#   TUYA_UID=               # optional, discovered automatically
#   TUYA_HOME_ID=           # optional, discovered automatically
_env_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env")
load_dotenv(_env_path, override=True)

# ── Configuration ─────────────────────────────────────────────────────────────
ACCESS_ID     = os.environ["TUYA_ACCESS_ID"]       # ~20 char alphanumeric
ACCESS_SECRET = os.environ["TUYA_ACCESS_KEY"]      # 32 char

# Regional API endpoint — must match your project's data center
# CN: https://openapi.tuyacn.com  EU: https://openapi.tuyaeu.com
# US: https://openapi.tuyaus.com  IN: https://openapi.tuyain.com
BASE_URL  = os.environ.get("TUYA_BASE_URL", "https://openapi.tuyaeu.com")

# Device to bind — from Device Management → your device
DEVICE_ID = os.environ["TUYA_DEVICE_ID"]
BIND_CODE = os.environ.get("TUYA_BIND_CODE", "")   # optional bind/authorization code

# SmartLife home to bind to — discovered automatically if not set
UID     = os.environ.get("TUYA_APP_USER_UID", "")
HOME_ID = os.environ.get("TUYA_HOME_ID", "")

print(f"ACCESS_ID : {ACCESS_ID[:4]}...{ACCESS_ID[-2:]}")
print(f"DEVICE_ID : {DEVICE_ID[:6]}...{DEVICE_ID[-2:]}")
print(f"BASE_URL  : {BASE_URL}")
print(f"UID       : {UID or '(will discover)'}")
print(f"HOME_ID   : {HOME_ID or '(will discover)'}")

In [ ]:
from tuya_connector import TuyaOpenAPI
from pprint import pprint

# Thin wrapper to keep the rest of the notebook unchanged
class TuyaAPI:
    def __init__(self, access_id: str, access_secret: str, base_url: str):
        self._api = TuyaOpenAPI(base_url, access_id, access_secret)

    def get_token(self):
        resp = self._api.connect()
        if not resp.get("success"):
            raise RuntimeError(f"Connect failed: {resp}")
        print(f"Token obtained OK")
        return resp

    def get(self, path: str) -> dict:
        return self._api.get(path)

    def post(self, path: str, body: dict) -> dict:
        return self._api.post(path, body)

    def put(self, path: str, body: dict) -> dict:
        return self._api.put(path, body)


api = TuyaAPI(ACCESS_ID, ACCESS_SECRET, BASE_URL)
api.get_token()

## Step 1 — Find your SmartLife UID

If you already know your UID, skip this cell and set `UID` in the config above.

Your SmartLife UID is also visible in:
- Tuya IoT Platform → Cloud → your project → Users tab
- Or by logging into SmartLife and checking Cloud → Devices → your account

In [ ]:
# ── Diagnostic: test which API endpoints are accessible ──
# If most fail with 1108 "uri path invalid", you need to subscribe to APIs:
#   Tuya IoT Platform → Cloud → your project → "Service API" tab
#   Subscribe: IoT Core, Smart Home Family Management, Device Status Notification

test_endpoints = [
    ("GET",  "/v1.0/token/uids",                           "Token UIDs"),
    ("GET",  f"/v1.0/devices/{DEVICE_ID}",                 "Device Info (v1)"),
    ("GET",  f"/v2.0/cloud/thing/{DEVICE_ID}",             "Device Info (v2 cloud)"),
    ("GET",  f"/v1.0/iot-03/devices/{DEVICE_ID}",          "Device Info (iot-03)"),
]

for method, ep, label in test_endpoints:
    resp = api.get(ep)
    status = "OK" if resp.get("success") else f"ERR {resp.get('code')}: {resp.get('msg')}"
    print(f"  [{status:40s}] {label:30s} {ep}")

## Step 2 — Find your SmartLife home ID

In [ ]:
if not UID:
    print("ERROR: Set UID first")
else:
    resp = api.get(f"/v1.0/users/{UID}/homes")
    if resp.get("success"):
        homes = resp["result"]
        print(f"Found {len(homes)} home(s):")
        for h in homes:
            print(f"  home_id={h['home_id']}  name={h['name']}  role={h.get('role', '?')}")
        if homes and not HOME_ID:
            HOME_ID = str(homes[0]["home_id"])
            print(f"\nUsing HOME_ID: {HOME_ID}")
    else:
        print("Failed:", resp)

## Step 3 — Check the device is reachable via API

In [ ]:
resp = api.get(f"/v1.0/devices/{DEVICE_ID}")
if resp.get("success"):
    d = resp["result"]
    print(f"Device found:")
    print(f"  id          : {d['id']}")
    print(f"  name        : {d.get('name','?')}")
    print(f"  product_id  : {d.get('product_id','?')}")
    print(f"  online      : {d.get('online', False)}")
    print(f"  uid         : {d.get('uid', '(not bound)')}")
    print(f"  home_id     : {d.get('home_id', '(not bound)')}")
else:
    print("Device not found or not accessible:", resp)

## Step 4 — Bind device to home

Two methods, try A first. If the device is already in the project (visible in Device Management)
but not bound to a home, Method B (direct user bind) is usually more reliable.

In [ ]:
# Method A — bind via authorization/bind code (if you have it)
# BIND_CODE comes from: Device Management → your device → Bind Code column
if not BIND_CODE:
    print("BIND_CODE not set — skipping Method A, use Method B below")
else:
    body = {"bind_code": BIND_CODE}
    resp = api.post(f"/v1.0/homes/{HOME_ID}/devices", body)
    print("Method A result:")
    pprint(resp)

In [ ]:
# Method B — directly assign device to user + home (works for pre-registered devices)
if not UID or not HOME_ID:
    print("ERROR: Set UID and HOME_ID first")
else:
    body = {"uid": UID, "home_id": int(HOME_ID)}
    resp = api.post(f"/v2.0/cloud/thing/{DEVICE_ID}/user", body)
    print("Method B result:")
    pprint(resp)
    if resp.get("success"):
        print("\nDevice bound successfully! Check SmartLife app.")
    else:
        print("\nBind failed. Try Method C below.")

In [ ]:
# Method C — transfer device ownership (alternative)
if not UID:
    print("ERROR: Set UID first")
else:
    body = {"uid": UID}
    resp = api.post(f"/v1.0/devices/{DEVICE_ID}/transfer", body)
    print("Method C result:")
    pprint(resp)

## Step 5 — Verify binding

In [ ]:
resp = api.get(f"/v1.0/devices/{DEVICE_ID}")
if resp.get("success"):
    d = resp["result"]
    print(f"Device status after bind attempt:")
    print(f"  uid     : {d.get('uid', '(not bound)')}")
    print(f"  home_id : {d.get('home_id', '(not bound)')}")
    print(f"  online  : {d.get('online', False)}")
    
    if d.get('uid') == UID:
        print("\nSUCCESS: Device is bound to your account!")
        print("Open SmartLife → Home → your home → the device should appear.")
    else:
        print("\nDevice still not bound to your UID.")
else:
    pprint(resp)

## Bonus — Send a property command to the device via OpenAPI

Even without SmartLife binding, you can send commands directly via the OpenAPI.
This is useful to verify the device is working end-to-end.

In [ ]:
# Send switch_1 = true to the device via Tuya OpenAPI
# This triggers the relay_status path on the ESP32 → HA MQTT
PROPERTY_NAME  = "relay_status_3"   # the property to set
PROPERTY_VALUE = True

body = {"properties": {PROPERTY_NAME: PROPERTY_VALUE}}
resp = api.post(f"/v2.0/cloud/thing/{DEVICE_ID}/shadow/properties/issue", body)
print(f"Send {PROPERTY_NAME}={PROPERTY_VALUE}:")
pprint(resp)

In [ ]:
# Read current property shadow state
resp = api.get(f"/v2.0/cloud/thing/{DEVICE_ID}/shadow/properties")
if resp.get("success"):
    props = resp["result"].get("properties", [])
    print(f"Current property states ({len(props)} properties):")
    for p in props:
        print(f"  {p['code']:30s} = {p['value']}  (time={p.get('time','?')})")
else:
    pprint(resp)